# Módulo 6 · Clase 4 — Estadística descriptiva

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Medidas de centralidad: media, mediana, moda |
| 2 | Medidas de dispersión: varianza, desviación estándar, rango |
| 3 | Percentiles, cuartiles y outliers |
| 4 | Forma de la distribución: asimetría y curtosis |
| 5 | Correlación |
| 6 | Visualización estadística |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Librerías listas")

---
## Motivación

```
Antes de entrenar cualquier modelo de IA:

1. ENTENDER los datos — ¿qué rango tienen? ¿hay outliers? ¿están sesgados?
2. LIMPIAR los datos — detectar errores, valores imposibles
3. PREPARAR los datos — normalizar, centrar, transformar

La estadística descriptiva es el lenguaje para hacer ese análisis.
```

In [ ]:
# Dataset: 50 estudiantes, edades, notas y horas de estudio
np.random.seed(42)
n = 50

edades  = np.random.normal(22, 3, n).clip(18, 35).round().astype(int)
horas   = np.random.normal(5, 1.5, n).clip(0.5, 10).round(1)
# Nota correlacionada con horas + ruido
notas   = (horas * 0.8 + np.random.normal(0, 1, n) + 2).clip(1, 10).round(1)
# Añadir un outlier
notas[0] = 0.5
horas[1] = 0.2

print("Dataset de 50 estudiantes")
print(f"Edades  (primeros 10): {edades[:10]}")
print(f"Horas   (primeros 10): {horas[:10]}")
print(f"Notas   (primeros 10): {notas[:10]}")

---
## 1. Medidas de centralidad

```
MEDIA:   suma de todos los valores / n
  ✓ Usa todos los datos
  ✗ Muy sensible a outliers

MEDIANA: valor central cuando ordenamos de menor a mayor
  ✓ Robusta frente a outliers
  ✓ Mejor para distribuciones asimétricas

MODA:    valor que aparece con más frecuencia
  ✓ Única métrica aplicable a datos categóricos

Ejemplo: salarios en una empresa pequeña
  [25000, 26000, 27000, 28000, 250000] (el CEO)
  Media:  71200 ← distorsionada por el CEO
  Mediana: 27000 ← más representativa
```

In [ ]:
print("=== Notas de los 50 estudiantes ===")
media   = np.mean(notas)
mediana = np.median(notas)
moda_r  = stats.mode(notas, keepdims=True)

print(f"Media:   {media:.4f}")
print(f"Mediana: {mediana:.4f}")
print(f"Moda:    {moda_r.mode[0]} (aparece {moda_r.count[0]} veces)")

print("\n=== Efecto de un outlier ===")
notas_con_outlier = np.append(notas, [0.5, 0.3, 0.1])  # 3 outliers extremos
print(f"Sin outliers extremos: media={media:.2f}, mediana={mediana:.2f}")
print(f"Con outliers extremos: media={np.mean(notas_con_outlier):.2f}, mediana={np.median(notas_con_outlier):.2f}")
print("→ La mediana apenas cambia; la media se desplaza notablemente")

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(notas, bins=15, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(media,   color='red',    ls='-',  lw=2, label=f'Media = {media:.2f}')
ax.axvline(mediana, color='orange', ls='--', lw=2, label=f'Mediana = {mediana:.2f}')
ax.set_xlabel('Nota'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de notas')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 2. Medidas de dispersión

```
RANGO:              max - min
VARIANZA (σ²):      media de las desviaciones al cuadrado
DESVIACIÓN ESTÁNDAR (σ): √varianza — mismas unidades que los datos
ERROR ESTÁNDAR:     σ / √n — precisión de la media estimada

Población vs Muestra:
  Varianza poblacional: σ² = Σ(xᵢ - μ)² / N      (ddof=0)
  Varianza muestral:    s² = Σ(xᵢ - x̄)² / (N-1)  (ddof=1) ← usar por defecto

¿Por N-1? Corrección de Bessel: la varianza muestral subestima
la varianza real si usamos N. N-1 corrige ese sesgo.
```

In [ ]:
print("=== Dispersión de notas ===")
rango    = np.max(notas) - np.min(notas)
var_pob  = np.var(notas, ddof=0)   # varianza poblacional
var_mues = np.var(notas, ddof=1)   # varianza muestral
std_pob  = np.std(notas, ddof=0)
std_mues = np.std(notas, ddof=1)
error_std= std_mues / np.sqrt(len(notas))

print(f"Mínimo:           {np.min(notas):.1f}")
print(f"Máximo:           {np.max(notas):.1f}")
print(f"Rango:            {rango:.1f}")
print(f"Varianza (N):     {var_pob:.4f}")
print(f"Varianza (N-1):   {var_mues:.4f}")
print(f"Desv. est. (N):   {std_pob:.4f}")
print(f"Desv. est. (N-1): {std_mues:.4f}")
print(f"Error estándar:   {error_std:.4f}")
print(f"\nInterpretación: La nota típica varía ±{std_mues:.2f} puntos respecto a la media ({media:.2f})")

# Comparar dos grupos con misma media, distinta dispersión
print("\n=== Dos clases con misma media, diferente dispersión ===")
clase_A = np.array([6.5, 7.0, 6.8, 7.2, 7.0, 6.9, 7.1, 7.0, 6.7, 7.3])
clase_B = np.array([3.0, 4.5, 5.5, 7.0, 8.5, 9.5, 7.5, 6.0, 5.0, 8.0])
print(f"Clase A: media={clase_A.mean():.2f}, std={clase_A.std():.2f} → notas muy homogéneas")
print(f"Clase B: media={clase_B.mean():.2f}, std={clase_B.std():.2f} → notas muy dispersas")

---
## 3. Percentiles, cuartiles y outliers

In [ ]:
print("=== Percentiles ===")
for p in [10, 25, 50, 75, 90, 95, 99]:
    val = np.percentile(notas, p)
    print(f"  P{p:2d}: {val:.2f}")

Q1  = np.percentile(notas, 25)
Q3  = np.percentile(notas, 75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f"\n=== Cuartiles y outliers ===")
print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = Q3 - Q1 = {IQR:.2f}")
print(f"Límites: [{limite_inf:.2f}, {limite_sup:.2f}]")

outliers = notas[(notas < limite_inf) | (notas > limite_sup)]
print(f"Outliers detectados: {outliers}")

# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(notas, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', lw=2),
                flierprops=dict(marker='o', color='red', ms=8))
axes[0].set_ylabel('Nota'); axes[0].set_title('Boxplot de notas')
axes[0].set_xticklabels(['Estudiantes'])

axes[1].boxplot([notas, clase_A, clase_B], vert=True, patch_artist=True,
                labels=['General', 'Clase A', 'Clase B'],
                boxprops=dict(facecolor='lightgreen'),
                medianprops=dict(color='red', lw=2))
axes[1].set_ylabel('Nota'); axes[1].set_title('Comparativa de grupos')

plt.tight_layout(); plt.show()

print("\n=== Descripción completa (numpy) ===")
resumen = {
    'n':      len(notas),
    'media':  round(np.mean(notas), 3),
    'std':    round(np.std(notas, ddof=1), 3),
    'min':    np.min(notas),
    'Q1':     round(Q1, 3),
    'mediana':round(np.median(notas), 3),
    'Q3':     round(Q3, 3),
    'max':    np.max(notas),
}
for k, v in resumen.items():
    print(f"  {k:10}: {v}")

---
## 4. Forma de la distribución

In [ ]:
print("=== Asimetría (skewness) ===")
print("  skewness = 0  → distribución simétrica")
print("  skewness > 0  → cola a la derecha (media > mediana)")
print("  skewness < 0  → cola a la izquierda (media < mediana)")

asimetria = stats.skew(notas)
curtosis  = stats.kurtosis(notas)
print(f"\nNotas: asimetría={asimetria:.4f}, curtosis={curtosis:.4f}")

# Distribuciones con distinta asimetría
np.random.seed(42)
simetrica    = np.random.normal(0, 1, 1000)
cola_derecha = np.random.exponential(1, 1000)          # cola a la derecha
cola_izquierda = -np.random.exponential(1, 1000)       # cola a la izquierda

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, datos, titulo in zip(axes,
    [simetrica, cola_derecha, cola_izquierda],
    ['Simétrica', 'Cola derecha (+)', 'Cola izquierda (-)']):
    ax.hist(datos, bins=40, density=True, alpha=0.7, color='steelblue')
    ax.axvline(np.mean(datos),   color='red',    ls='-', lw=2, label='Media')
    ax.axvline(np.median(datos), color='orange', ls='--',lw=2, label='Mediana')
    skew = stats.skew(datos)
    ax.set_title(f"{titulo}\nskewness={skew:.2f}")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 5. Correlación

```
La correlación de Pearson mide la relación lineal entre dos variables.

r ∈ [-1, 1]
  r = 1   → relación lineal positiva perfecta
  r = -1  → relación lineal negativa perfecta
  r = 0   → sin relación lineal

IMPORTANTE: correlación ≠ causalidad
  Helados vendidos correlaciona con ahogados → causa: el calor
  La cigüeña correlaciona con nacimientos → ambas dependen de la ruralidad
```

In [ ]:
r_pearson, p_valor = stats.pearsonr(horas, notas)
print(f"Correlación horas↔notas: r={r_pearson:.4f}, p={p_valor:.4f}")
print(f"Significativa (p<0.05): {p_valor < 0.05}")

# Matriz de correlación
datos_matriz = np.column_stack([edades, horas, notas])
nombres_vars = ['Edad', 'Horas estudio', 'Nota']
corr_matrix  = np.corrcoef(datos_matriz.T)

print("\nMatriz de correlación:")
header = f"{'':15}" + "".join(f"{n:>15}" for n in nombres_vars)
print(header)
for i, nombre in enumerate(nombres_vars):
    fila = f"{nombre:15}" + "".join(f"{corr_matrix[i,j]:>15.4f}" for j in range(3))
    print(fila)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter horas vs notas
axes[0].scatter(horas, notas, alpha=0.6, color='steelblue')
m, b = np.polyfit(horas, notas, 1)
x_line = np.linspace(horas.min(), horas.max(), 100)
axes[0].plot(x_line, m*x_line + b, 'r-', lw=2)
axes[0].set_xlabel('Horas de estudio'); axes[0].set_ylabel('Nota')
axes[0].set_title(f'Horas vs Nota (r={r_pearson:.3f})')
axes[0].grid(True, alpha=0.3)

# Heatmap de correlación
im = axes[1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
axes[1].set_xticklabels(nombres_vars, rotation=45, ha='right')
axes[1].set_yticklabels(nombres_vars)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f"{corr_matrix[i,j]:.2f}", ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Matriz de correlación')

plt.tight_layout(); plt.show()

# Correlación no implica causalidad
print("\n=== Correlaciones engañosas ===")
np.random.seed(42)
x_ruido = np.random.randn(50)
y_cuad  = x_ruido**2 + np.random.randn(50) * 0.1
r_cuad, _ = stats.pearsonr(x_ruido, y_cuad)
print(f"Relación cuadrática y=x²: r_pearson={r_cuad:.4f} (¡casi 0! pero hay relación clara)")
print("→ Pearson solo detecta relaciones LINEALES")

---
## Práctica

### Ejercicio 1 — Análisis de salarios

**Enunciado:** Dado este dataset de salarios anuales (€):

```python
salarios = [28000, 32000, 35000, 31000, 29000, 38000, 34000,
            33000, 30000, 36000, 29500, 37000, 31500, 350000,
            32500, 28500, 39000, 33500, 30500, 36500]
```

1. Calcula media, mediana y moda
2. Calcula desviación estándar e IQR
3. Detecta outliers con el método IQR
4. Compara la media con y sin outliers
5. ¿Cuál es el salario que supera al 80% de la muestra?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
from scipy import stats

salarios = np.array([28000, 32000, 35000, 31000, 29000, 38000, 34000,
                     33000, 30000, 36000, 29500, 37000, 31500, 350000,
                     32500, 28500, 39000, 33500, 30500, 36500])

# 1.
media   = np.mean(salarios)
mediana = np.median(salarios)
moda    = stats.mode(salarios, keepdims=True)
print(f"Media:   {media:,.0f}€")
print(f"Mediana: {mediana:,.0f}€")
print(f"Moda:    {moda.mode[0]:,.0f}€ (todos únicos → no hay moda clara)")

# 2.
std = np.std(salarios, ddof=1)
Q1  = np.percentile(salarios, 25)
Q3  = np.percentile(salarios, 75)
IQR = Q3 - Q1
print(f"\nDesv. estándar: {std:,.0f}€")
print(f"IQR: {IQR:,.0f}€  (Q1={Q1:,.0f}, Q3={Q3:,.0f})")

# 3. Outliers
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = salarios[(salarios < lim_inf) | (salarios > lim_sup)]
print(f"\nLímites: [{lim_inf:,.0f}, {lim_sup:,.0f}]")
print(f"Outliers: {outliers}")

# 4.
sin_outliers = salarios[(salarios >= lim_inf) & (salarios <= lim_sup)]
print(f"\nMedia CON outlier:    {np.mean(salarios):,.0f}€")
print(f"Media SIN outlier:    {np.mean(sin_outliers):,.0f}€")
print(f"Diferencia: {np.mean(salarios) - np.mean(sin_outliers):,.0f}€")

# 5.
p80 = np.percentile(salarios, 80)
print(f"\nPercentil 80: {p80:,.0f}€")
```
</details>

---
### Ejercicio 2 — Comparación de grupos

**Enunciado:** Tres modelos de ML producen estas tasas de error en 20 ejecuciones:

```python
np.random.seed(0)
modelo_A = np.random.normal(0.15, 0.02, 20)
modelo_B = np.random.normal(0.13, 0.05, 20)
modelo_C = np.random.normal(0.14, 0.01, 20)
```

1. Para cada modelo: media, mediana, std, IQR
2. ¿Cuál es el mejor modelo? ¿Solo mirar la media es suficiente?
3. Visualiza los tres con boxplots y violín plots
4. ¿Cuál tiene menor variabilidad? ¿Por qué importa la variabilidad en producción?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
modelo_A = np.random.normal(0.15, 0.02, 20)
modelo_B = np.random.normal(0.13, 0.05, 20)
modelo_C = np.random.normal(0.14, 0.01, 20)
modelos  = {'A': modelo_A, 'B': modelo_B, 'C': modelo_C}

# 1. Métricas
for nombre, m in modelos.items():
    Q1, Q3 = np.percentile(m, [25, 75])
    print(f"Modelo {nombre}: media={m.mean():.4f}, std={m.std(ddof=1):.4f}, "
          f"mediana={np.median(m):.4f}, IQR={Q3-Q1:.4f}")

# 2. Análisis
print("\nB tiene la menor media PERO la mayor variabilidad (std=0.05)")
print("C tiene media intermedia pero la menor variabilidad (std=0.01)")
print("→ En producción: C puede ser más fiable aunque no sea el más preciso en promedio")

# 3. Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

data = [modelo_A, modelo_B, modelo_C]
axes[0].boxplot(data, labels=['A','B','C'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', lw=2))
axes[0].set_title('Boxplot - Error por modelo'); axes[0].set_ylabel('Tasa de error')
axes[0].grid(True, alpha=0.3)

axes[1].violinplot(data, positions=[1,2,3], showmedians=True)
axes[1].set_xticks([1,2,3]); axes[1].set_xticklabels(['A','B','C'])
axes[1].set_title('Violin plot'); axes[1].set_ylabel('Tasa de error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 3 — Normalización

**Enunciado:** Implementa y compara tres métodos de normalización:

```
Min-Max (escala [0,1]):  x' = (x - min) / (max - min)
Z-score (estándar):      x' = (x - μ) / σ
Robusto (resistente a outliers): x' = (x - mediana) / IQR
```

Aplícalos a `[2, 5, 3, 8, 100, 4, 6, 3, 7, 5]` y compara el efecto de cada método con y sin el outlier (100).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np

datos = np.array([2, 5, 3, 8, 100, 4, 6, 3, 7, 5], dtype=float)

def min_max(x):  return (x - x.min()) / (x.max() - x.min())
def zscore(x):   return (x - x.mean()) / x.std(ddof=1)
def robusto(x):
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    return (x - med) / iqr

print("Con outlier (100):")
print(f"{'Valor':>8} {'Min-Max':>10} {'Z-score':>10} {'Robusto':>10}")
for orig, mm, zs, rob in zip(datos, min_max(datos), zscore(datos), robusto(datos)):
    print(f"{orig:>8.0f} {mm:>10.4f} {zs:>10.4f} {rob:>10.4f}")

datos_sin = datos[datos != 100]
print("\nSin outlier:")
print(f"{'Valor':>8} {'Min-Max':>10} {'Z-score':>10} {'Robusto':>10}")
for orig, mm, zs, rob in zip(datos_sin, min_max(datos_sin), zscore(datos_sin), robusto(datos_sin)):
    print(f"{orig:>8.0f} {mm:>10.4f} {zs:>10.4f} {rob:>10.4f}")

print("\nConclusión:")
print("  Min-Max: muy afectado por outliers (comprime el rango normal)")
print("  Z-score: moderadamente afectado")
print("  Robusto: casi no cambia → ideal cuando hay outliers")
```
</details>

---
### Ejercicio 4 — Análisis completo de un dataset

**Enunciado:** Genera y analiza este dataset de consumo energético de edificios:

```python
np.random.seed(7)
n = 100
superficie  = np.random.normal(250, 80, n).clip(50, 600)
ocupantes   = np.random.poisson(15, n).clip(1, 50)
antiguedad  = np.random.uniform(0, 40, n)
consumo_kwh = (superficie * 1.2 + ocupantes * 200 + antiguedad * 50
               + np.random.normal(0, 2000, n)).clip(500)
```

1. Calcula estadísticas descriptivas para todas las variables
2. Matriz de correlación y heatmap
3. ¿Qué variable tiene más correlación con el consumo?
4. Detecta edificios con consumo anómalamente alto (outliers)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)
n = 100
superficie  = np.random.normal(250, 80, n).clip(50, 600)
ocupantes   = np.random.poisson(15, n).clip(1, 50).astype(float)
antiguedad  = np.random.uniform(0, 40, n)
consumo     = (superficie * 1.2 + ocupantes * 200 + antiguedad * 50
               + np.random.normal(0, 2000, n)).clip(500)

vars_dict = {'Superficie': superficie, 'Ocupantes': ocupantes,
             'Antigüedad': antiguedad, 'Consumo kWh': consumo}

# 1. Descriptivas
print(f"{'Variable':<15} {'Media':>10} {'Std':>10} {'Min':>10} {'Med':>10} {'Max':>10}")
for nombre, v in vars_dict.items():
    print(f"{nombre:<15} {v.mean():>10.1f} {v.std(ddof=1):>10.1f} {v.min():>10.1f} {np.median(v):>10.1f} {v.max():>10.1f}")

# 2. Correlación
X = np.column_stack(list(vars_dict.values()))
C = np.corrcoef(X.T)
nombres = list(vars_dict.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(4)); axes[0].set_yticks(range(4))
axes[0].set_xticklabels(nombres, rotation=45, ha='right')
axes[0].set_yticklabels(nombres)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f"{C[i,j]:.2f}", ha='center', va='center')
plt.colorbar(im, ax=axes[0]); axes[0].set_title('Correlación')

axes[1].boxplot([(consumo - np.median(consumo)) / (np.percentile(consumo,75)-np.percentile(consumo,25))],
                vert=False, patch_artist=True, boxprops=dict(facecolor='lightblue'),
                flierprops=dict(marker='o', color='red', ms=8))
axes[1].set_title('Consumo normalizado (outliers en rojo)'); axes[1].set_xlabel('Score robusto')
plt.tight_layout(); plt.show()

# 3. Mayor correlación con consumo
idx_consumo = 3
for i, nombre in enumerate(nombres[:-1]):
    print(f"r({nombre}, Consumo) = {C[i, idx_consumo]:.4f}")

# 4. Outliers en consumo
Q1, Q3 = np.percentile(consumo, [25, 75])
IQR = Q3 - Q1
outliers_idx = np.where((consumo > Q3 + 1.5*IQR))[0]
print(f"\nEdificios con consumo anómalo: {len(outliers_idx)}")
for i in outliers_idx:
    print(f"  Edificio {i}: {consumo[i]:.0f} kWh (sup={superficie[i]:.0f}m², ocup={ocupantes[i]:.0f})")
```
</details>

---
### Ejercicio 5 (reto) — Dashboard estadístico

**Enunciado:** Implementa una función `resumen_estadistico(datos, nombre)` que:

1. Calcula todas las métricas descriptivas
2. Detecta outliers
3. Clasifica la distribución (simétrica/asimétrica izq/asimétrica der)
4. Genera un panel de 4 gráficas: histograma+KDE, boxplot, QQ-plot, histograma acumulado

Pruébalo con los datos de consumo, notas y salarios de los ejercicios anteriores.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def resumen_estadistico(datos, nombre="Variable"):
    datos = np.array(datos, dtype=float)
    n = len(datos)
    media, mediana = np.mean(datos), np.median(datos)
    std   = np.std(datos, ddof=1)
    Q1, Q3 = np.percentile(datos, [25, 75])
    IQR   = Q3 - Q1
    skew  = stats.skew(datos)
    kurt  = stats.kurtosis(datos)
    _, p_norm = stats.shapiro(datos[:50] if n > 50 else datos)

    # Clasificar forma
    if abs(skew) < 0.5:   forma = "Aproximadamente simétrica"
    elif skew > 0:         forma = "Asimétrica derecha (cola +)"
    else:                  forma = "Asimétrica izquierda (cola -)"

    outliers = datos[(datos < Q1-1.5*IQR) | (datos > Q3+1.5*IQR)]

    print(f"{'='*50}")
    print(f" {nombre} (n={n})")
    print(f"{'='*50}")
    print(f" Media:    {media:.4f}  | Mediana: {mediana:.4f}")
    print(f" Std:      {std:.4f}  | IQR:     {IQR:.4f}")
    print(f" Min:      {datos.min():.4f}  | Max:     {datos.max():.4f}")
    print(f" Asimetría:{skew:.4f}  | Curtosis:{kurt:.4f}")
    print(f" Forma:    {forma}")
    print(f" Normal (Shapiro p={p_norm:.4f}): {'Sí' if p_norm>0.05 else 'No'}")
    print(f" Outliers: {len(outliers)}  {outliers[:5]}")

    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    fig.suptitle(nombre, fontsize=11)

    # Histograma + KDE
    axes[0].hist(datos, bins='auto', density=True, alpha=0.6, color='steelblue')
    kde = stats.gaussian_kde(datos)
    x_kde = np.linspace(datos.min(), datos.max(), 200)
    axes[0].plot(x_kde, kde(x_kde), 'r-', lw=2)
    axes[0].axvline(media, color='orange', ls='--', lw=1.5, label='media')
    axes[0].axvline(mediana, color='green', ls=':', lw=1.5, label='mediana')
    axes[0].legend(fontsize=7); axes[0].set_title('Histograma + KDE')

    # Boxplot
    axes[1].boxplot(datos, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', lw=2))
    axes[1].set_title('Boxplot')

    # QQ-plot
    stats.probplot(datos, plot=axes[2])
    axes[2].set_title('QQ-plot (vs Normal)')

    # CDF empírica
    x_sorted = np.sort(datos)
    cdf = np.arange(1, n+1) / n
    axes[3].step(x_sorted, cdf, where='post', color='steelblue', lw=2)
    axes[3].set_title('CDF empírica')
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

# Probar
np.random.seed(42)
resumen_estadistico(np.random.normal(7, 1.5, 50).clip(0, 10), "Notas")
```
</details>

---
## Resumen

```python
import numpy as np
from scipy import stats

x = np.array([...])

# Centralidad
np.mean(x); np.median(x); stats.mode(x)

# Dispersión
np.std(x, ddof=1)          # desviación estándar muestral
np.var(x, ddof=1)          # varianza muestral
np.percentile(x, [25,75])  # Q1, Q3
# IQR = Q3 - Q1

# Outliers
outliers = x[(x < Q1 - 1.5*IQR) | (x > Q3 + 1.5*IQR)]

# Forma
stats.skew(x)              # asimetría
stats.kurtosis(x)          # curtosis

# Correlación
np.corrcoef(x, y)          # matriz de correlación
stats.pearsonr(x, y)       # r + p-valor
```

## ¿Qué viene en la próxima clase?
En la **Clase 5** damos el salto a la inferencia estadística: ¿cómo sacamos conclusiones sobre una población a partir de una muestra?